# Lecture 3: NumPy arrays, Matplotlib, pandas, and GitHub workflow

**PHYS690: Computational Methods for Physics Research**  
**Thursday, September 3, 2026**

Today we will build a compact scientific Python workflow using tools that will show up constantly in computational physics work:

- **NumPy** for numerical arrays and vectorized calculation.
- **pandas** for tabular data, CSV files, and quick data summaries.
- **Matplotlib** for figures.
- **Git/GitHub** for tracking work and sharing reproducible project changes.

The theme is simple: start with arrays, organize those arrays into a table, write and read a CSV file, make a figure, move a calculation into a script that can be rerun from the terminal, then push your local changes to a remote GitHub repository branch.

## How to use this notebook

This notebook is intended to run on your laptop in **VS Code**, not Google Colab.

Before running the notebook, open this repository in VS Code and activate your course environment in the VS Code terminal.

On macOS/Linux:

```bash
source .venv/bin/activate
```

On Windows using Git Bash:

```bash
source .venv/Scripts/activate
```

If you have not installed the packages yet, run this while the environment is active:

```bash
pip install numpy matplotlib pandas ipykernel jupyter
```

Then select the `.venv` Python kernel in VS Code. This notebook creates files under `scratch/lecture03/`, which is intentionally ignored by Git in this course repository.

## Resources for today

- Course setup: [VS Code resource](../resources/vscode.md)
- Course computing overview: [Unit 1 research computing resource](../resources/unit-1-research-computing.md)
- NumPy: [absolute beginner's guide](https://numpy.org/doc/stable/user/absolute_beginners)
- pandas: [read_csv reference](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)
- Matplotlib: [getting started guide](https://matplotlib.org/stable/users/getting_started/)
- GitHub Docs: [pushing commits to a remote repository](https://docs.github.com/en/get-started/using-git/pushing-commits-to-a-remote-repository)

You do not need to memorize these pages. The goal is to know what each tool is for, recognize common syntax, and practice looking up details when you need them.

## Learning goals

By the end of this lecture, you should be able to:

- Create and inspect NumPy arrays using `np.array`, `np.linspace`, `.shape`, `.dtype`, and indexing.
- Use vectorized array operations instead of writing a loop for every calculation.
- Build a pandas `DataFrame`, save it to CSV, read it back, and add derived columns.
- Create labeled Matplotlib figures and save them as files.
- Recognize when a calculation belongs in a notebook and when it should become a reusable script.
- Use common Git commands to check status, create a branch, commit local changes, and push that branch to GitHub.

About one third of today will be hands-on coding or terminal workflow practice. When you see an **In-class coding activity**, pause, edit the notebook or terminal command yourself, and compare results with a neighbor before we discuss as a group.

# Part 1: Imports and scratch workspace

A typical scientific Python notebook begins by importing the tools we need. The standard short names are conventions used across the Python scientific ecosystem:

| Package | Common import | Main role today |
| --- | --- | --- |
| NumPy | `import numpy as np` | Arrays and numerical calculations |
| pandas | `import pandas as pd` | Tables and CSV files |
| Matplotlib | `import matplotlib.pyplot as plt` | Plots and figures |

We will also use a few standard-library tools: `Path` for file paths, `os` for setting the working directory, `sys` for the active Python executable, and `subprocess` for running a script from inside the notebook.

In [ ]:
%matplotlib inline

from pathlib import Path
import os
import subprocess
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# VS Code usually starts notebooks from the workspace root.
# This fallback handles the common case where the notebook starts in lectures/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "lectures":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Use the repository root so notebook paths and terminal paths match.
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd()

# Keep generated lecture files in scratch/, which is ignored by Git.
SCRATCH_DIR = PROJECT_ROOT / "scratch" / "lecture03"
DATA_DIR = SCRATCH_DIR / "data"
FIGURE_DIR = SCRATCH_DIR / "figures"
SCRIPT_DIR = SCRATCH_DIR / "scripts"

# A list is an ordered Python container; here each entry is a Path object.
for directory in [DATA_DIR, FIGURE_DIR, SCRIPT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Python executable: {sys.executable}")
print(f"Working directory: {PROJECT_ROOT}")
print(f"NumPy {np.__version__} | pandas {pd.__version__}")
print(f"Matplotlib {matplotlib.__version__}")


### In-class coding activity 1: verify the same environment, 4 minutes

In the VS Code terminal, with your `.venv` active, run:

```bash
pwd
ls scratch/lecture03
python -c "import sys; print(sys.executable)"
```

Compare the terminal output with the notebook output above. This checks that your terminal and notebook are using the same project and Python environment.

# Part 2: NumPy arrays

A Python **list** is a flexible container that can hold many kinds of objects. A NumPy **array** is more specialized: it is designed for many numerical values with the same data type. That specialization is why NumPy can make numerical code concise and fast.

Common syntax: from [NumPy beginner's guide](https://numpy.org/doc/stable/user/absolute_beginners)

| Task | Syntax | Meaning |
| --- | --- | --- |
| Create an array | `np.array([1, 2, 3])` | Convert a Python sequence into an array |
| Evenly spaced values | `np.linspace(0, 1, 11)` | 11 values from 0 to 1, including endpoints |
| Integer range | `np.arange(0, 10, 2)` | Values 0, 2, 4, 6, 8 |
| Shape | `array.shape` | Size along each dimension |
| Data type | `array.dtype` | Type stored in the array |
| First value | `array[0]` | Indexing starts at 0 |
| Last value | `array[-1]` | Negative indices count from the end |
| Boolean mask | `array[array > 0]` | Select values satisfying a condition |


In [ ]:
# A Python list is a general-purpose container.
time_list = [0, 1, 2, 3, 4]
print(type(time_list), time_list)

# A NumPy array supports vectorized numerical operations.
time_array = np.array(time_list, dtype=float)
print(type(time_array), time_array)

# Vectorized arithmetic applies the operation to every element.
print("time_array + 0.5 =", time_array + 0.5)
print("time_array**2 =", time_array**2)

# Arrays carry useful metadata about shape, size, and type.
print("shape:", time_array.shape)
print("size:", time_array.size)
print("dtype:", time_array.dtype)

Now we will create a small synthetic position measurement, similar to our last lecture:

$$
x(t) = A e^{-t/	au} + c + \epsilon.
$$

Here `A` is an amplitude, `tau` is a decay time, `c` is an offset, and `epsilon` represents measurement noise. This is not meant to be a full model-fitting lesson yet; it is a convenient dataset for practicing arrays, tables, plots, file output, and scripts.

In [ ]:
# A random number generator gives reproducible random values when seeded.
rng = np.random.default_rng(seed=690)

# np.linspace creates evenly spaced time values for the measurement.
time_s = np.linspace(0.0, 10.0, 21)

# These scalar values define the synthetic physical model.
amplitude_true = 1.0
tau_true = 3.00
offset_true = 0.00

# NumPy evaluates the exponential model for every time value at once.
clean_position = amplitude_true * np.exp(-time_s / tau_true) + offset_true

# Each point gets the same uncertainty in this simple example.
position_error = np.full_like(time_s, 0.025)

# Random normal noise turns the ideal curve into measured data.
measured_position = clean_position + rng.normal(loc=0.0, scale=position_error)

# column_stack combines several 1D arrays into one 2D array.
measurements = np.column_stack((time_s, measured_position, position_error))

# raw 2D array output
print(measurements)

print("measurements shape:", measurements.shape)
print("first row:", measurements[0])
print("last row:", measurements[-1])

### In-class coding activity 2: NumPy slicing and masks, 6 minutes

Run the cell below once, then edit it to answer these questions with code:

- What is the mean measured position for times `time_s >= 8.0`?
- How many measured positions are below `0.30`?
- What times and positions are selected by the window `2.0 <= time_s <= 6.0`?

Use Boolean masks and NumPy reductions. Avoid manually counting values from printed output.

In [ ]:
# Slicing with : selects a range of entries from an array.
first_five_positions = measured_position[:5]
print("first five positions:", first_five_positions)

# A Boolean mask is an array of True/False values used for selection.
late_time_mask = time_s >= 6.0
late_time_positions = measured_position[late_time_mask]

# Reductions summarize many values into one value.
print("number of late-time points:", late_time_positions.size)
print("mean late-time position:", late_time_positions.mean())
print("standard deviation:", late_time_positions.std(ddof=1))

# Activity tasks: edit and complete the lines below.
very_late_mask = ...
mean_after_8 = ...

below_threshold_mask = ...
n_below_threshold = ...

window_mask = ...
window_times = ...
window_positions = ...

print("mean position after 8 s:", mean_after_8)
print("number of points below 0.30:", n_below_threshold)
print("times from 2 to 6 s:", window_times)
print("positions from 2 to 6 s:", window_positions)


# Part 3: pandas tables and CSV files

NumPy arrays are excellent for numerical calculation. pandas is useful when the data naturally has column names, row labels, mixed data types, or needs to move in and out of files.

The main pandas object today is the [**DataFrame**](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html), which is a table with labeled columns. A single column from a DataFrame is a **Series**.

Common syntax:

| Task | Syntax | Meaning |
| --- | --- | --- |
| Create table | `pd.DataFrame(dictionary)` | Build a table from named columns |
| First rows | `df.head()` | Preview the data |
| Column names | `df.columns` | Inspect available columns |
| Select column | `df["position"]` | Return one Series |
| Add column | `df["residual"] = ...` | Store a derived quantity |
| Save CSV | `df.to_csv(path, index=False)` | Write a table to disk |
| Read CSV | `pd.read_csv(path)` | Load a CSV as a DataFrame |
| Summary stats | `df.describe()` | Quick numerical summary |


In [ ]:
# A dictionary stores key-value pairs inside curly braces.
# Here each key becomes a column name and each value is an array.
decay_dictionary = {
    "time_s": time_s,
    "position": measured_position,
    "position_error": position_error,
    "model_position": clean_position,
}

# pandas converts the dictionary into a labeled table.
decay_df = pd.DataFrame(decay_dictionary)

# head() displays the first rows without printing the whole table.
decay_df.head()

In [ ]:
# Save the generated table to a CSV file inside scratch/.
csv_path = DATA_DIR / "synthetic_decay.csv"
decay_df.to_csv(csv_path, index=False)

# Read the CSV back from disk, just as we would with experimental data.
loaded_df = pd.read_csv(csv_path)

# Display the path and a compact preview of the loaded table.
print(f"Wrote and reloaded: {csv_path}")
loaded_df.head()

In [ ]:
# A DataFrame column can be used in arithmetic much like a NumPy array.
loaded_df["residual"] = loaded_df["position"] - loaded_df["model_position"]

# A normalized residual measures the residual in units of the uncertainty.
loaded_df["normalized_residual"] = loaded_df["residual"] / loaded_df["position_error"]

# np.where chooses labels based on a condition: early times or late times.
loaded_df["time_region"] = np.where(loaded_df["time_s"] < 6.0, "early", "late")

# aggregate computes summary statistics for the whole table or for selected columns.
print(loaded_df.agg({"position": ["mean", "std"], "residual": ["mean", "std"]}))

# groupby splits the table by label before computing summary statistics.
loaded_df.groupby("time_region")[["position", "residual"]].agg(["mean", "std"])

### In-class coding activity 3: inspect and extend the table, 5 minutes

Use the cell below as a sandbox. Complete these tasks with pandas code:

- Display `loaded_df.describe()`.
- Print the column names.
- Create a new column called `position_squared`.
- Create a new column called `fractional_error = position_error / position`.
- Select only the rows with `time_s > 4.0` and display the first few rows.
- Compute the mean `position` and `residual` for each `time_region`.

In [ ]:
# Sandbox cell for DataFrame practice.
# Complete each task from the activity prompt.

# Display summary statistics for the numerical columns.
...

# Print the column names.
...

# Add derived columns.
loaded_df["position_squared"] = ...
loaded_df["fractional_error"] = ...

# Select rows later than 4 seconds.
late_rows = ...
late_rows.head()

# Compute grouped means by time region.
...


# Part 4: Matplotlib figures

Matplotlib organizes plots around a **Figure** and one or more **Axes** objects. The figure is the whole canvas. An axes object is one plotting region with x and y coordinates.  See the [getting started guide](https://matplotlib.org/stable/users/getting_started/) for more examples.


Common syntax:

| Task | Syntax | Meaning |
| --- | --- | --- |
| Create figure and axes | `fig, ax = plt.subplots()` | Start a plot |
| Plot line | `ax.plot(x, y)` | Draw connected values |
| Plot points with error bars | `ax.errorbar(x, y, yerr=err, fmt="o")` | Draw measurements and uncertainty |
| Axis label | `ax.set_xlabel("time (s)")` | Label the x-axis |
| Legend | `ax.legend()` | Show labels from plotted data |
| Save figure | `fig.savefig(path, dpi=200)` | Write figure to disk |


In [ ]:
# Create one figure with one axes object.
fig, ax = plt.subplots(figsize=(7.0, 4.5))

# Plot measured positions with vertical error bars.
ax.errorbar(
    loaded_df["time_s"],
    loaded_df["position"],
    yerr=loaded_df["position_error"],
    fmt="o",
    color="blue",
    ecolor="blue",
    capsize=3,
    label="synthetic measurements",
)

# Overlay the clean model used to generate the data.
ax.plot(
    loaded_df["time_s"],
    loaded_df["model_position"],
    color="orange",
    linewidth=2,
    label="generating model",
)

# Labels, title, grid, and legend make the figure interpretable.
ax.set_xlabel("time (s)")
ax.set_ylabel("position: x(t)")
ax.set_title("Synthetic exponential relaxation dataset")
ax.grid(alpha=0.25)
ax.legend()

# Save the figure so it can be included in a repository or report.
decay_plot_path = FIGURE_DIR / "synthetic_decay_with_model.png"
fig.savefig(decay_plot_path)

print(f"Saved figure to: {decay_plot_path}")

In [ ]:
# Subplots let us place related views side by side.
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0))

# The first axes shows position versus time.
axes[0].scatter(loaded_df["time_s"], loaded_df["position"], color="#1f6f78")
axes[0].set_xlabel("time (s)")
axes[0].set_ylabel("position: x(t)")
axes[0].set_title("Measured positions")

# The second axes shows whether residuals are centered near zero.
axes[1].hist(loaded_df["normalized_residual"], bins=8, color="#6f8f3a", edgecolor="white")
axes[1].set_xlabel("normalized residual")
axes[1].set_ylabel("count")
axes[1].set_title("Residual check")

# tight_layout reduces overlapping labels.
fig.tight_layout()

summary_plot_path = FIGURE_DIR / "synthetic_decay_summary.png"
fig.savefig(summary_plot_path)

print(f"Saved figure to: {summary_plot_path}")

### In-class coding activity 4: create and save your own figure, 7 minutes

Make a new figure using `loaded_df` and save it to:

```text
scratch/lecture03/figures/student_decay_plot.png
```

Your figure must include labeled axes, a title, and a legend. Make at least one intentional visual choice different from the examples above, such as marker style, color, line style, or which subset of the data you plot.

In [ ]:
# Student figure activity.
# Create and save scratch/lecture03/figures/student_decay_plot.png.

student_plot_path = FIGURE_DIR / "student_decay_plot.png"

fig, ax = ...

# TODO: Add at least one plotted data series from loaded_df.
...

# TODO: Add axis labels, a title, and a legend.
...

# TODO: Save the figure to student_plot_path.
...

print(f"Saved student figure to: {student_plot_path}")


### Figure expectations for assignments and projects

Every saved figure should include enough information that someone can understand it without opening your notebook code. At minimum, include:

- clearly labeled x- and y-axes, including units when relevant;
- a title or caption-level description of what is being compared;
- a legend when more than one data set, model, or curve is shown;
- uncertainty bars when the plotted values have measurement or statistical uncertainties;
- readable marker sizes, line widths, and font sizes;
- a filename and repository location that make the figure easy to find again.

For assignments and projects, also explain in the README or nearby text what input data produced the figure and what command or notebook cells recreate it.

# Part 5: From notebook cells to a Python script

A notebook is excellent for exploration, but a script is better when you want one command to repeat the same workflow. Here we write a small script that reads the CSV file created earlier and saves a plot. This script does not depend on previous notebook variables; it starts from an input file on disk.

In [ ]:
%%writefile scratch/lecture03/scripts/analyze_decay.py
"""Analyze a synthetic decay CSV file and save a plot.

Usage:
    python scratch/lecture03/scripts/analyze_decay.py scratch/lecture03/data/synthetic_decay.csv scratch/lecture03/figures/script_decay_plot.png
"""

from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd


def main(input_csv, output_figure):
    """Load data from a CSV file and save a labeled plot."""
    data = pd.read_csv(input_csv)

    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    ax.errorbar(
        data["time_s"],
        data["position"],
        yerr=data["position_error"],
        fmt="o",
        capsize=3,
        label="synthetic measurements",
    )
    ax.plot(data["time_s"], data["model_position"], label="generating model")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("position: x(t)")
    ax.set_title("Script output: synthetic decay data")
    ax.grid(alpha=0.25)
    ax.legend()

    output_figure = Path(output_figure)
    output_figure.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_figure, dpi=200, bbox_inches="tight")

    print(f"Read input CSV: {input_csv}")
    print(f"Saved figure to: {output_figure}")


if __name__ == "__main__":
    if len(sys.argv) != 3:
        raise SystemExit("Usage: python analyze_decay.py input.csv output.png")

    main(Path(sys.argv[1]), Path(sys.argv[2]))


In [ ]:
# Run the script using the same Python executable as this notebook kernel.
script_path = SCRIPT_DIR / "analyze_decay.py"
script_output_path = FIGURE_DIR / "script_decay_plot.png"

# subprocess.run starts a command-line process from Python.
completed = subprocess.run(
    [sys.executable, str(script_path), str(csv_path), str(script_output_path)],
    cwd=PROJECT_ROOT,
    check=True,
    text=True,
    capture_output=True,
)

# Print the script output so we can see what happened.
print(completed.stdout)


### In-class coding activity 5: run and modify the script, 5 minutes

In the VS Code terminal, make sure your `.venv` is active and you are at the repository root. Then run:

```bash
python scratch/lecture03/scripts/analyze_decay.py scratch/lecture03/data/synthetic_decay.csv scratch/lecture03/figures/script_decay_plot.png
```

Then inspect the output files:

```bash
ls scratch/lecture03/data
ls scratch/lecture03/figures
```

Now make one small script modification: change the plot title or line color, rerun the command, and confirm that `script_decay_plot.png` was updated. This is the beginning of a reproducible command-line workflow: one command reads an input file, performs an analysis, and creates an output figure.

# Part 6: Common Git and GitHub workflow

A reproducible workflow is not only about Python. After creating a script, you need to keep track of what changed and push that work somewhere your collaborators or instructor can see it. In this course, that remote location is GitHub.

Common commands:

```bash
git status
git branch
git switch -c your-username-lecture03
git add path/to/file
git commit -m "Short message describing the change"
git push -u origin your-username-lecture03
```

A useful habit is to run `git status` before and after every commit. It tells you which files changed, which files are staged, and whether your local branch has commits that need to be pushed.

### In-class coding activity 6: push your script branch to GitHub, 8 minutes

In this activity, you will commit the notebook changes and the script you created in Part 5, then push them to GitHub on a branch named with your username.

1. In the notebook cell below, set `GITHUB_USERNAME` to your GitHub username and run the cell.
2. In the VS Code terminal, create and switch to a branch named `your-username-lecture03`.
3. Save this notebook after your edits.
4. Stage the changed notebook and force-add `scratch/lecture03/scripts/analyze_decay.py`, since `scratch/` is normally ignored by Git.
5. Commit and push the branch to GitHub.

Use your actual GitHub username in place of `your-username`. If your branch already exists, use `git switch your-username-lecture03` instead of `git switch -c ...`.

In [ ]:
# Set this to your GitHub username, then run the cell.
GITHUB_USERNAME = "replace-with-your-username"
BRANCH_NAME = f"{GITHUB_USERNAME}-lecture03"
SCRIPT_PATH = "scratch/lecture03/scripts/analyze_decay.py"
NOTEBOOK_PATH = "lectures/lecture03_9_3_26_numpy_scipy_matplotlib_pandas.ipynb"

print("Use this branch name:", BRANCH_NAME)
print()
print("Suggested terminal commands:")
print("git status")
print(f"git switch -c {BRANCH_NAME}")
print(f"git add {NOTEBOOK_PATH}")
print(f"git add -f {SCRIPT_PATH}")
print("git status")
print("git commit -m 'Complete Lecture 03 script workflow practice'")
print(f"git push -u origin {BRANCH_NAME}")
print("git status")


After pushing, open the repository on GitHub and confirm that your branch appears in the branch menu. Check that the commit includes the notebook and `scratch/lecture03/scripts/analyze_decay.py`. This is the same pattern you will use for assignment work: edit locally, commit locally, push to GitHub, and then open a pull request if requested.

# Part 7: Putting the tools together

A practical workflow often looks like this:

| Step | Tool | Example from today |
| --- | --- | --- |
| Generate or calculate numerical values | NumPy | `np.linspace`, `np.exp`, array arithmetic |
| Organize values as columns | pandas | `pd.DataFrame`, derived columns |
| Save and reload inputs | pandas | `to_csv`, `read_csv` |
| Visualize the result | Matplotlib | `fig, ax = plt.subplots()`, `ax.errorbar` |
| Make the analysis rerunnable | Python script | `python analyze_decay.py input.csv output.png` |
| Share reproducible changes | Git/GitHub | `git status`, `git add -f`, `git commit`, `git push` |

As your projects become larger, the details will change, but this loop will keep reappearing.